# ACCIDENT V7 — V5 recipe from the self-supervised VideoMAE-base (KAGGLE)
Targets LOCKED before any run: `baselines/2026-10-05_m8_accident_model/TARGETS_VIDEO_V7_LOCKED.md`. One change vs V5: the backbone starts from `MCG-NJU/videomae-base` (masked-video pretraining, no Kinetics labels), stage 1 runs 6 epochs. V7 replaces V4 only if H1-H5 all pass.

**Setup (right panel):** *Add Input* -> dataset `picekl/accident` only (no weights needed). Settings -> Accelerator **GPU T4 x2**, **Internet ON** (downloads the backbone). Then *Save Version -> Save & Run All (Commit)* (~6-8 h, limit 12 h). Result: Output tab -> `accident_v7_results.zip` (send it back).


In [ ]:
!pip -q install transformers==5.19.0 timm opencv-python-headless pandas scikit-learn scipy
import torch, os, transformers; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '', '| CPUs', os.cpu_count(), '| transformers', transformers.__version__)
assert transformers.__version__ == '5.19.0'
os.environ['ACC_VID'] = 'MCG-NJU/videomae-base'   # BEFORE importing m8_accident_video (read at import)


In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/metadata-real.csv', recursive=True)
assert hits, 'Add the dataset picekl/accident with Add Input (right panel)'
ROOT_DIR = os.path.dirname(hits[0]); print(ROOT_DIR, os.listdir(ROOT_DIR)[:10])

In [ ]:
%%writefile m8_accident_train_v2.py
"""ACCIDENT-benchmark training + one-shot scoring (see TARGETS_LOCKED.md).

Per-frame EfficientNet-B0 (normal vs accident), a k-of-n persistence alarm, thresholds chosen on a
validation split of the TRAIN clips, each test split scored once. Frames before the impact time are
"normal", frames after are "accident"; the +-window around the impact is ignored when training.
"""
from __future__ import annotations
import json, os, pathlib, random, re, time
from multiprocessing import Pool
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn

SIZE = int(os.environ.get("ACC_SIZE", 384)); FPS = 2.0
MODEL = os.environ.get("ACC_MODEL", "efficientnet_b2")
PREV = 2                                    # previous frame = 1 s earlier at 2 fps
GAP_BEFORE_S, GAP_AFTER_S = 1.0, 0.5       # ignored around the impact when training
DETECT_WITHIN_S = 10.0                      # B1 window
CFG_GRID = [(k, n) for k, n in ((2, 3), (3, 5), (3, 4), (4, 6))]
THR_GRID = [round(x, 2) for x in np.arange(0.30, 0.96, 0.05)]
VAL_EARLY_CAP = 0.04                        # margin under the locked B2 bar of 5 %


# ---------------------------------------------------------------- frames
def sample_frames(path, fps=FPS, size=SIZE):
    cap = cv2.VideoCapture(str(path)); src = cap.get(cv2.CAP_PROP_FPS) or 15.0
    step = max(1, int(round(src / fps))); frames, times, i = [], [], 0
    while cap.grab():
        if i % step == 0:
            ok, f = cap.retrieve()
            if ok:
                frames.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (size, size),
                                         interpolation=cv2.INTER_AREA))
                times.append(i / src)
        i += 1
    cap.release()
    if not frames:
        return np.zeros((0, size, size, 3), np.uint8), np.zeros(0)
    return np.stack(frames), np.array(times)


def _extract(args):
    src, dst = args
    if os.path.exists(dst + ".npz"):
        return dst
    fr, t = sample_frames(src)
    np.savez(dst + ".npz", f=fr, t=t)
    return dst


def extract_all(items, cache_dir, workers=None):
    """items: list of (video_path, key). Returns {key: npz path}."""
    pathlib.Path(cache_dir).mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(pathlib.Path(cache_dir) / k)) for p, k in items]
    with Pool(workers or os.cpu_count() or 2) as pool:
        for n, _ in enumerate(pool.imap_unordered(_extract, jobs, chunksize=4)):
            if n % 200 == 0:
                print(f"  extracted {n}/{len(jobs)}", flush=True)
    return {k: str(pathlib.Path(cache_dir) / (k + ".npz")) for _, k in items}


def clip_key(path):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(path))


def group_id(path):
    return pathlib.Path(path).stem[:11]       # YouTube id: clips of one source video share it


# ---------------------------------------------------------------- data
def build_store(parts, cache, out_path):
    """parts: [(df, path_col, is_synth)] -> disk-backed uint8 store of ALL frames (N,H,W,3), labels
    (-1 ignored / 0 normal / 1 accident), index of each frame's previous frame, synthetic flag."""
    lens = []
    for df, col, _ in parts:
        for _, r in df.iterrows():
            lens.append(len(np.load(cache[clip_key(r[col])])["t"]))
    mm = np.lib.format.open_memmap(out_path, mode="w+", dtype=np.uint8, shape=(sum(lens), SIZE, SIZE, 3))
    y, prev, syn, pos = [], [], [], 0
    for df, col, is_syn in parts:
        for _, r in df.iterrows():
            z = np.load(cache[clip_key(r[col])]); fr, t = z["f"], z["t"]
            ta = float(r["accident_time"]); lab = np.full(len(t), -1)
            lab[t < ta - GAP_BEFORE_S] = 0
            lab[t >= ta + GAP_AFTER_S] = 1
            mm[pos:pos + len(t)] = fr
            y.append(lab); syn.append(np.full(len(t), is_syn))
            prev.append(pos + np.maximum(0, np.arange(len(t)) - PREV)); pos += len(t)
    mm.flush()
    return mm, np.concatenate(y), np.concatenate(prev), np.concatenate(syn)


MEAN = torch.tensor([0.485, 0.456, 0.406] * 2).view(1, 6, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225] * 2).view(1, 6, 1, 1)


class Net(nn.Module):
    """6-channel input: current RGB + RGB one second earlier, in [0,1], NCHW -> logit.
    Normalisation is inside, so the ONNX takes [0,1] pixels."""
    def __init__(self, pretrained=True):
        super().__init__()
        import timm
        self.m = timm.create_model(MODEL, pretrained=pretrained, num_classes=1, in_chans=6)
        self.register_buffer("mean", MEAN.clone()); self.register_buffer("std", STD.clone())

    def forward(self, x):
        return self.m((x - self.mean) / self.std).squeeze(1)


def stack_t(cur, prev, device):
    x = np.concatenate([cur, prev], axis=3)
    return torch.from_numpy(x).to(device).permute(0, 3, 1, 2).float() / 255.0


def augment(x):
    if random.random() < 0.5:
        x = x.flip(3)
    b = 1 + (torch.rand(x.shape[0], 1, 1, 1, device=x.device) - 0.5) * 0.5
    c = 1 + (torch.rand(x.shape[0], 1, 1, 1, device=x.device) - 0.5) * 0.5
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    return ((x - m) * c + m * b).clamp(0, 1)


def train(store, use, epochs=10, bs=32, lr=2e-4, pretrained=True, device=None, seed=0, log=print):
    """store=(mm,y,prev,syn); use = boolean mask over frames to train on (labelled ones only)."""
    mm, y, prev, _ = store
    pool = np.nonzero(use & (y >= 0))[0]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    net = Net(pretrained).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * max(1, len(pool) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    yy = y[pool]
    pw = torch.tensor((yy == 0).sum() / max(1, (yy == 1).sum()), device=device).float()
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(pool); tot = 0.0; nb = 0
        for s in range(0, len(perm) - bs + 1, bs):
            b = np.sort(perm[s:s + bs])
            xb = augment(stack_t(mm[b], mm[prev[b]], device)); yb = torch.from_numpy(y[b]).float().to(device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                loss = lossf(net(xb), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss); nb += 1
        log(f"  epoch {ep + 1}/{epochs} loss {tot / max(1, nb):.4f}")
    return net


@torch.no_grad()
def clip_probs(net, df, cache, path_col, device=None, bs=64):
    device = device or next(net.parameters()).device
    net.eval(); out = {}
    for _, r in df.iterrows():
        z = np.load(cache[clip_key(r[path_col])]); fr, t = z["f"], z["t"]
        pi = np.maximum(0, np.arange(len(fr)) - PREV)
        p = np.concatenate([torch.sigmoid(net(stack_t(fr[i:i + bs], fr[pi[i:i + bs]], device))).cpu().numpy()
                            for i in range(0, len(fr), bs)]) if len(fr) else np.zeros(0)
        out[r[path_col]] = (t, p)
    return out


# ---------------------------------------------------------------- alarm + metrics
def first_alarm_times(t, p, thr, k, n):
    """Times at which >= k of the last n frames are >= thr (every alarming frame)."""
    hot = (p >= thr).astype(int); out = []
    for i in range(len(hot)):
        if hot[max(0, i - n + 1):i + 1].sum() >= k:
            out.append(t[i])
    return out


def evaluate(probs, df, path_col, thr, k, n):
    rows = []
    for _, r in df.iterrows():
        t, p = probs[r[path_col]]; ta = float(r["accident_time"])
        al = first_alarm_times(t, p, thr, k, n)
        early = any(a < ta for a in al)
        post = [a for a in al if ta <= a <= ta + DETECT_WITHIN_S]
        rows.append({"path": r[path_col], "early": early, "detected": bool(post),
                     "delay": (post[0] - ta) if post else None, "ta": ta})
    d = pd.DataFrame(rows)
    det = d["detected"].mean(); early = d["early"].mean()
    strict = (d["detected"] & ~d["early"]).mean()
    delays = d["delay"].dropna()
    return {"n": len(d), "B1_detect_within_10s": float(det), "B2_early_alarm": float(early),
            "strict_detect_no_early": float(strict),
            "B3_median_delay_s": float(delays.median()) if len(delays) else None}, d


def choose_config(probs, df, path_col):
    best = None
    for k, n in CFG_GRID:
        for thr in THR_GRID:
            m, _ = evaluate(probs, df, path_col, thr, k, n)
            feas = m["B2_early_alarm"] <= VAL_EARLY_CAP
            key = (feas, m["B1_detect_within_10s"] if feas else -m["B2_early_alarm"],
                   -(m["B3_median_delay_s"] or 99))
            if best is None or key > best[0]:
                best = (key, {"thr": thr, "k": k, "n": n, "val": m, "feasible": feas})
    return best[1]


def breakdown(d, df, path_col, cols=("day_time", "weather", "scene_layout", "quality", "type")):
    m = d.merge(df, left_on="path", right_on=path_col)
    out = {}
    for c in cols:
        if c in m:
            out[c] = {str(v): {"n": int(len(g)), "detect": round(float(g["detected"].mean()), 3),
                               "early": round(float(g["early"].mean()), 3)}
                      for v, g in m.groupby(c)}
    return out


def split_train_val(df, path_col, frac=0.2, seed=0):
    g = sorted({group_id(p) for p in df[path_col]}); rng = random.Random(seed); rng.shuffle(g)
    val = set(g[:max(1, int(len(g) * frac))])
    isval = df[path_col].map(lambda p: group_id(p) in val)
    return df[~isval].copy(), df[isval].copy()


def export_onnx(net, path):
    net.eval().cpu()
    kw = dict(input_names=["image"], output_names=["logit"],
              dynamic_axes={"image": {0: "b"}, "logit": {0: "b"}}, opset_version=17)
    try:
        torch.onnx.export(net, torch.rand(1, 6, SIZE, SIZE), path, dynamo=False, **kw)
    except TypeError:   # older torch has no `dynamo` argument
        torch.onnx.export(net, torch.rand(1, 6, SIZE, SIZE), path, **kw)


# ---------------------------------------------------------------- protocol
def run_protocol(real, syn, cache, split_col, out_dir, synth_n=300, epochs=10, pretrained=True,
                 seed=0, bs=32, log=print):
    """real/syn: metadata frames whose clips are already in `cache`. Trains real-only and
    real+synthetic, picks the variant + alarm config on the validation clips, then scores the
    split's test clips ONCE. Returns the results dict (also written to out_dir)."""
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    tr_all = real[real[split_col] == "train"]; te = real[real[split_col] == "test"]
    tr, va = split_train_val(tr_all, "path", seed=seed)
    log(f"[{split_col}] train {len(tr)} val {len(va)} test {len(te)} clips")
    syn_s = syn.sample(n=min(synth_n, len(syn)), random_state=seed) if len(syn) else syn
    parts = [(tr, "path", False)] + ([(syn_s, "rgb_path", True)] if len(syn_s) else [])
    store = build_store(parts, cache, str(out / f"store_{split_col}.npy"))
    mm, y, prev, synflag = store
    log(f"  frames: real {int(((y >= 0) & ~synflag).sum())} labelled (acc {int(((y == 1) & ~synflag).sum())}), "
        f"synthetic {int(((y >= 0) & synflag).sum())}")
    variants, picks = {}, {}
    for name, use in {"real_only": ~synflag, "real_plus_synth": np.ones(len(y), bool)}.items():
        if name == "real_plus_synth" and not synflag.any():
            continue
        log(f" training {name} ({int((use & (y >= 0)).sum())} frames)")
        net = train(store, use, epochs=epochs, bs=bs, pretrained=pretrained, seed=seed, log=log)
        pv = clip_probs(net, va, cache, "path")
        cfg = choose_config(pv, va, "path")
        log(f"  val: {cfg['val']}  thr {cfg['thr']} k/n {cfg['k']}/{cfg['n']} feasible {cfg['feasible']}")
        variants[name] = net; picks[name] = cfg
    del mm
    os.remove(out / f"store_{split_col}.npy")
    def score_key(c):
        return (c["feasible"], c["val"]["B1_detect_within_10s"] if c["feasible"] else -c["val"]["B2_early_alarm"])
    chosen = max(picks, key=lambda n: score_key(picks[n]))
    cfg = picks[chosen]
    (out / f"chosen_{split_col}.json").write_text(json.dumps({"variant": chosen, **cfg, "picks": picks}, indent=1, default=str))
    log(f" CHOSEN on validation: {chosen} {cfg['thr']} {cfg['k']}/{cfg['n']} -- config frozen, scoring test once")
    net = variants[chosen]
    pt = clip_probs(net, te, cache, "path")
    m, d = evaluate(pt, te, "path", cfg["thr"], cfg["k"], cfg["n"])
    res = {"split": split_col, "variant": chosen, "thr": cfg["thr"], "k": cfg["k"], "n": cfg["n"],
           "val": cfg["val"], "test": m, "breakdown": breakdown(d, te, "path"),
           "val_picks": {k: v["val"] for k, v in picks.items()}}
    (out / f"results_{split_col}.json").write_text(json.dumps(res, indent=1, default=str))
    torch.save(net.state_dict(), out / f"accident_v2_{split_col}.pt")
    return res, net


def verdict(res_in, res_geo):
    a, b = res_in["test"], res_geo["test"]
    rows = [("B1 in-dist detect<=10s >=70%", a["B1_detect_within_10s"], a["B1_detect_within_10s"] >= 0.70),
            ("B2 in-dist early alarm <=5%", a["B2_early_alarm"], a["B2_early_alarm"] <= 0.05),
            ("B3 median delay <=5s", a["B3_median_delay_s"], (a["B3_median_delay_s"] is not None and a["B3_median_delay_s"] <= 5)),
            ("B4a geo detect >=60%", b["B1_detect_within_10s"], b["B1_detect_within_10s"] >= 0.60),
            ("B4b geo early alarm <=8%", b["B2_early_alarm"], b["B2_early_alarm"] <= 0.08)]
    return rows


In [ ]:
%%writefile m8_accident_video.py
"""V3: fine-tune a pretrained video backbone (VideoMAE) on ACCIDENT windows (see TARGETS_VIDEO_LOCKED.md).
Reuses the v2 protocol helpers (grouped validation split, k-of-n alarm, locked metrics)."""
from __future__ import annotations
import json, os, pathlib, random
from multiprocessing import Pool
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
import m8_accident_train_v2 as A

FPS, SIZE, NF, STRIDE = 4.0, 224, 16, 4          # 16 frames at 4 fps = 4-s window; scored every 4 frames = 1 s
GAP_BEFORE_S, GAP_AFTER_S, POS_MAX_S = 1.0, 0.5, 3.0
MODEL_ID = os.environ.get("ACC_VID", "MCG-NJU/videomae-base-finetuned-kinetics")
A.CFG_GRID = [(1, 1), (2, 3), (2, 4), (3, 5)]    # persistence in units of 1-s windows
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)


# ------------------------------------------------------------ frames cache (4 fps, 224 px)
def _extract(args):
    src, dst = args
    if os.path.exists(dst + ".npz"): return dst
    cap = cv2.VideoCapture(src); sfps = cap.get(cv2.CAP_PROP_FPS) or 15.0; step = max(1, int(round(sfps / FPS))); fr, ts, i = [], [], 0
    while cap.grab():
        if i % step == 0:
            ok, f = cap.retrieve()
            if ok: fr.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (SIZE, SIZE), interpolation=cv2.INTER_AREA)); ts.append(i / sfps)
        i += 1
    cap.release()
    np.savez(dst + ".npz", f=np.stack(fr) if fr else np.zeros((0, SIZE, SIZE, 3), np.uint8), t=np.array(ts))
    return dst


def extract_all(items, cache_dir):
    pathlib.Path(cache_dir).mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(pathlib.Path(cache_dir) / k)) for p, k in items]
    with Pool(os.cpu_count() or 2) as pool:
        for n, _ in enumerate(pool.imap_unordered(_extract, jobs, chunksize=4)):
            if n % 200 == 0: print(f"  extracted {n}/{len(jobs)}", flush=True)
    return {k: str(pathlib.Path(cache_dir) / (k + ".npz")) for _, k in items}


# ------------------------------------------------------------ windows + labels (pure)
def win_idx(e, n=None):
    return np.clip(np.arange(e - NF + 1, e + 1), 0, None)


def window_ends(n):
    return list(range(NF - 1, n, STRIDE))


def window_label(te, ta):
    if te <= ta - GAP_BEFORE_S: return 0
    if ta + GAP_AFTER_S <= te <= ta + POS_MAX_S: return 1
    return -1


def sample_windows(t, ta, rng, n_pos=4, n_neg=4):
    """Training windows for one clip: any frame as the window end (not just the 1-s grid)."""
    ends = list(range(NF - 1, len(t)))
    pos = [e for e in ends if window_label(t[e], ta) == 1]; neg = [e for e in ends if window_label(t[e], ta) == 0]
    P = rng.sample(pos, min(n_pos, len(pos))) if pos else []; N = rng.sample(neg, min(n_neg, len(neg))) if neg else []
    return [(e, 1) for e in P] + [(e, 0) for e in N]


class ClipDS(torch.utils.data.Dataset):
    def __init__(self, df, cache, seed=0): self.df, self.cache, self.seed = df.reset_index(drop=True), cache, seed
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; z = np.load(self.cache[A.clip_key(r["path"])]); fr, t = z["f"], z["t"]
        rng = random.Random(hash((self.seed, i, random.random())))
        ws = sample_windows(t, float(r["accident_time"]), rng) if len(fr) >= NF else []
        if not ws: return torch.zeros(0, NF, SIZE, SIZE, 3, dtype=torch.uint8), torch.zeros(0)
        x = np.stack([fr[win_idx(e)] for e, _ in ws]); return torch.from_numpy(x), torch.tensor([y for _, y in ws], dtype=torch.float32)


def collate(b):
    xs = [x for x, _ in b if len(x)]; ys = [y for _, y in b if len(y)]
    return (torch.cat(xs), torch.cat(ys)) if xs else (None, None)


# ------------------------------------------------------------ model
def build_model():
    from transformers import VideoMAEForVideoClassification
    m = VideoMAEForVideoClassification.from_pretrained(MODEL_ID, num_labels=1, ignore_mismatched_sizes=True)
    b = m.videomae.encoder.layer[0].attention.attention.value.bias       # older transformers 5.x drop the checkpoint's q/v biases silently
    if b is not None and float(b.abs().sum()) == 0 and not os.environ.get("ACC_ALLOW_BIAS_RESET"):
        raise RuntimeError("Pretrained VideoMAE q/v biases were NOT loaded (old transformers). Run: pip install -U transformers, restart the session, re-run.")
    m.gradient_checkpointing_enable()          # 8 windows x 1568 tokens would not fit a 16-GB T4 otherwise
    return m


def to_input(x, device, train=False):
    x = x.to(device).permute(0, 1, 4, 2, 3).float() / 255.0          # B,T,C,H,W
    if train:
        flip = torch.rand(x.shape[0], device=device) < 0.5; x = torch.where(flip.view(-1, 1, 1, 1, 1), x.flip(-1), x)
        x = (x * (0.8 + 0.4 * torch.rand(x.shape[0], 1, 1, 1, 1, device=device))).clamp(0, 1)
    return (x - MEAN.to(device)) / STD.to(device)


def logits(model, x):
    o = model(pixel_values=x); o = getattr(o, "logits", o); return o.view(o.shape[0])


def train(model, df, cache, epochs=4, lr=5e-5, clips_per_batch=1, device=None, log=print, seed=0):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu"); model.to(device).train()
    dl = torch.utils.data.DataLoader(ClipDS(df, cache, seed), batch_size=clips_per_batch, shuffle=True, num_workers=2, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05); steps = epochs * len(dl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=device == "cuda"); lossf = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        tot, n = 0.0, 0
        for x, y in dl:
            if x is None: sched.step(); continue
            with torch.autocast(device_type=device if device == "cuda" else "cpu", enabled=device == "cuda"):
                loss = lossf(logits(model, to_input(x, device, True)).float(), y.to(device))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(y); n += len(y)
        log(f"  epoch {ep + 1}/{epochs} loss {tot / max(1, n):.4f} ({n} windows)")
    return model


@torch.no_grad()
def clip_probs(model, df, cache, device=None, bs=16):
    device = device or next(model.parameters()).device; model.eval(); out = {}
    for _, r in df.iterrows():
        z = np.load(cache[A.clip_key(r["path"])]); fr, t = z["f"], z["t"]; ends = window_ends(len(fr)); ps = []
        for i in range(0, len(ends), bs):
            x = torch.from_numpy(np.stack([fr[win_idx(e)] for e in ends[i:i + bs]]))
            with torch.autocast(device_type="cuda", enabled=str(device).startswith("cuda")):
                ps.append(torch.sigmoid(logits(model, to_input(x, device)).float()).cpu().numpy())
        out[r["path"]] = (np.array([t[e] for e in ends]), np.concatenate(ps) if ps else np.zeros(0))
    return out


# ------------------------------------------------------------ protocol
def run_protocol(real, cache, split_col, out_dir, epochs=4, seed=0, log=print, model_fn=build_model):
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    tr_all = real[real[split_col] == "train"]; te = real[real[split_col] == "test"]
    tr, va = A.split_train_val(tr_all, "path", seed=seed); log(f"[{split_col}] train {len(tr)} val {len(va)} test {len(te)}")
    torch.manual_seed(seed); model = train(model_fn(), tr, cache, epochs=epochs, log=log, seed=seed)
    cfg = A.choose_config(clip_probs(model, va, cache), va, "path")
    log(f" CHOSEN on validation: thr {cfg['thr']} k/n {cfg['k']}/{cfg['n']} feasible {cfg['feasible']} val {cfg['val']}")
    (out / f"chosen_{split_col}.json").write_text(json.dumps(cfg, indent=1, default=str))
    m, d = A.evaluate(clip_probs(model, te, cache), te, "path", cfg["thr"], cfg["k"], cfg["n"])      # test scored ONCE
    res = {"split": split_col, "thr": cfg["thr"], "k": cfg["k"], "n": cfg["n"], "val": cfg["val"], "test": m, "breakdown": A.breakdown(d, te, "path")}
    (out / f"results_{split_col}.json").write_text(json.dumps(res, indent=1, default=str)); return res, model


In [ ]:
%%writefile m8_accident_video4.py
"""V4: motion-proposed local crops + VideoMAE (see TARGETS_VIDEO_V4_LOCKED.md). Reuses V3 training/protocol code."""
from __future__ import annotations
import os, pathlib, random
from multiprocessing import Pool
import cv2, numpy as np, pandas as pd, torch
import m8_accident_video as V
import m8_accident_train_v2 as A

FPS, NF, SIZE = V.FPS, V.NF, V.SIZE
MAXSIDE, GRID, CROP_FRAC, K_LOCAL = 960, 32, 0.4, 2


# ------------------------------------------------------------ cache: JPEG frames + motion maps
def _extract(args):
    src, dst = args
    if os.path.exists(dst + ".npz"): return dst
    cap = cv2.VideoCapture(src); sfps = cap.get(cv2.CAP_PROP_FPS) or 15.0; step = max(1, int(round(sfps / FPS)))
    bufs, ts, mm, prev, i, hw = [], [], [], None, 0, (0, 0)
    while cap.grab():
        if i % step == 0:
            ok, f = cap.retrieve()
            if ok:
                h, w = f.shape[:2]; s = min(1.0, MAXSIDE / max(h, w))
                if s < 1: f = cv2.resize(f, (max(2, int(w * s)), max(2, int(h * s))), interpolation=cv2.INTER_AREA)
                hw = f.shape[:2]
                g = cv2.GaussianBlur(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), (64, 64), interpolation=cv2.INTER_AREA), (3, 3), 0).astype(np.float32)
                mm.append(np.zeros((GRID, GRID), np.float32) if prev is None else cv2.resize(np.abs(g - prev), (GRID, GRID), interpolation=cv2.INTER_AREA)); prev = g
                bufs.append(cv2.imencode(".jpg", f, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].ravel()); ts.append(i / sfps)
        i += 1
    cap.release()
    lens = np.array([len(b) for b in bufs], np.int64)
    np.savez(dst + ".npz", blob=np.concatenate(bufs) if bufs else np.zeros(0, np.uint8), off=np.concatenate([[0], np.cumsum(lens)]),
             t=np.array(ts), m=np.stack(mm).astype(np.float16) if mm else np.zeros((0, GRID, GRID), np.float16), hw=np.array(hw))
    return dst


def extract_all(items, cache_dir):
    pathlib.Path(cache_dir).mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(pathlib.Path(cache_dir) / k)) for p, k in items]
    with Pool(os.cpu_count() or 2) as pool:
        for n, _ in enumerate(pool.imap_unordered(_extract, jobs, chunksize=4)):
            if n % 200 == 0: print(f"  extracted {n}/{len(jobs)}", flush=True)
    return {k: str(pathlib.Path(cache_dir) / (k + ".npz")) for _, k in items}


class Clip:
    def __init__(s, path):
        z = np.load(path); s.blob, s.off, s.t = z["blob"], z["off"], z["t"]; s.h, s.w = [int(v) for v in z["hw"]]
        s.cs = np.cumsum(z["m"].astype(np.float32), 0); s._fc = {}
    def n(s): return len(s.t)
    def frame(s, i):
        if i not in s._fc: s._fc[i] = cv2.imdecode(s.blob[s.off[i]:s.off[i + 1]], cv2.IMREAD_COLOR)[:, :, ::-1]
        return s._fc[i]
    def heat(s, e): return s.cs[e] - s.cs[e - NF + 1]


# ------------------------------------------------------------ proposals + views
def crop_side(w, h): return max(8, int(CROP_FRAC * min(w, h)))


def proposals(heat, w, h, k=K_LOCAL):
    S = crop_side(w, h); sx, sy = S / w, S / h; hm = cv2.blur(heat.astype(np.float32), (3, 3)).copy(); xs = (np.arange(GRID) + .5) / GRID; out = []
    for _ in range(k):
        j = np.unravel_index(np.argmax(hm), hm.shape)
        if hm[j] <= 0: break
        cy, cx = xs[j[0]], xs[j[1]]; out.append((float(np.clip(cx, sx / 2, 1 - sx / 2)), float(np.clip(cy, sy / 2, 1 - sy / 2))))
        hm[(np.abs(xs[None, :] - cx) <= sx) & (np.abs(xs[:, None] - cy) <= sy)] = 0
    return out


def covers(view, crash, w, h):
    if view is None or crash is None or not np.isfinite(crash).all(): return False
    S = crop_side(w, h); return abs(crash[0] - view[0]) <= S / w / 2 and abs(crash[1] - view[1]) <= S / h / 2


def make_view(c, e, view):
    fr = [c.frame(i) for i in V.win_idx(e)]
    if view is None: return np.stack([cv2.resize(f, (SIZE, SIZE), interpolation=cv2.INTER_AREA) for f in fr])
    S = crop_side(c.w, c.h); x0 = int(np.clip(round(view[0] * c.w - S / 2), 0, c.w - S)); y0 = int(np.clip(round(view[1] * c.h - S / 2), 0, c.h - S))
    return np.stack([cv2.resize(f[y0:y0 + S, x0:x0 + S], (SIZE, SIZE), interpolation=cv2.INTER_AREA) for f in fr])


def proposal_coverage(df, cache, n=300, seed=0):
    d = df.sample(min(len(df), n), random_state=seed); rng = random.Random(seed); hit, frac, chance = [], [], []
    for _, r in d.iterrows():
        c = Clip(cache[A.clip_key(r["path"])]); ta = float(r["accident_time"]); crash = (float(r["center_x"]), float(r["center_y"]))
        ends = [e for e in range(NF - 1, c.n()) if V.window_label(c.t[e], ta) == 1]
        if not ends: continue
        cov = [any(covers(v, crash, c.w, c.h) for v in proposals(c.heat(e), c.w, c.h)) for e in ends]
        rnd = [any(covers((rng.random(), rng.random()), crash, c.w, c.h) for _ in range(K_LOCAL)) for _ in ends]
        hit.append(any(cov)); frac.append(np.mean(cov)); chance.append(np.mean(rnd))
    return dict(clips=len(hit), clip_has_covering_crop=float(np.mean(hit)), windows_covered=float(np.mean(frac)), windows_covered_by_random_crops=float(np.mean(chance)))


# ------------------------------------------------------------ training data + inference
class ClipDS4(torch.utils.data.Dataset):
    def __init__(self, df, cache, seed=0): self.df, self.cache, self.seed = df.reset_index(drop=True), cache, seed
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; c = Clip(self.cache[A.clip_key(r["path"])]); ta = float(r["accident_time"]); rng = random.Random(hash((self.seed, i, random.random())))
        if c.n() < NF: return torch.zeros(0, NF, SIZE, SIZE, 3, dtype=torch.uint8), torch.zeros(0)
        crash = (float(r["center_x"]), float(r["center_y"])); xs, ys = [], []
        for e, y in V.sample_windows(c.t, ta, rng):
            locs = proposals(c.heat(e), c.w, c.h)
            if y == 1:
                cov = [v for v in locs if covers(v, crash, c.w, c.h)]; v = rng.choice(cov + [None]) if cov else None
            else: v = rng.choice([None] + locs)
            xs.append(make_view(c, e, v)); ys.append(y)
        return (torch.from_numpy(np.stack(xs)), torch.tensor(ys, dtype=torch.float32)) if xs else (torch.zeros(0, NF, SIZE, SIZE, 3, dtype=torch.uint8), torch.zeros(0))


@torch.no_grad()
def clip_probs(model, df, cache, device=None, bs=12):
    device = device or next(model.parameters()).device; model.eval(); out = {}
    for _, r in df.iterrows():
        c = Clip(cache[A.clip_key(r["path"])]); ends = V.window_ends(c.n()); items, owner = [], []
        for k, e in enumerate(ends):
            for v in [None] + proposals(c.heat(e), c.w, c.h): items.append(make_view(c, e, v)); owner.append(k)
        p = np.zeros(len(ends), np.float32)
        for i in range(0, len(items), bs):
            with torch.autocast(device_type="cuda", enabled=str(device).startswith("cuda")):
                q = torch.sigmoid(V.logits(model, V.to_input(torch.from_numpy(np.stack(items[i:i + bs])), device)).float()).cpu().numpy()
            np.maximum.at(p, np.array(owner[i:i + bs]), q)
        out[r["path"]] = (np.array([c.t[e] for e in ends]), p); c._fc.clear()
    return out


def install():
    V.ClipDS = ClipDS4; V.clip_probs = clip_probs


def run_protocol(real, cache, split_col, out_dir, **kw):
    install(); return V.run_protocol(real, cache, split_col, out_dir, **kw)


In [ ]:
%%writefile m8_accident_false_alarm.py
"""Pure helpers for the V4 false-alarm proxy and parity check (see TARGETS_VIDEO_V4_FALSE_ALARM_LOCKED.md)."""
from __future__ import annotations
import math
import numpy as np


def hot_now(p, i, thr, k, n, delta=0.0):
    """True if at window i at least k of the last n scores are >= thr (and, when delta > 0, also at least delta above the
    clip's recent baseline: the median of the 8 scores before that n-window, if there are >= 3 of them)."""
    p = np.asarray(p, float); lo = max(0, i - n + 1); win = p[lo:i + 1]
    if delta > 0:
        hist = p[max(0, lo - 8):lo]; base = float(np.median(hist)) if len(hist) >= 3 else 0.0
        hot = (win >= thr) & (win - base >= delta)
    else:
        hot = win >= thr
    return int(hot.sum()) >= k


def alarm_runs(times, p, thr, k, n, delta=0.0):
    """Alarm times: the window end at which the rule first holds and no run was open (same as AccidentAlarm)."""
    runs, open_ = [], False
    for i, t in enumerate(times):
        if hot_now(p, i, thr, k, n, delta):
            if not open_: runs.append(float(t)); open_ = True
        else: open_ = False
    return runs


def pre_impact(times, p, ta, rule, gap=1.0):
    """(exposure seconds, runs before impact - gap, any alarm before impact) for one clip."""
    times = np.asarray(times, float); p = np.asarray(p, float)
    runs = alarm_runs(times, p, rule["thr"], rule["k"], rule["n"], rule.get("delta", 0.0))
    exposure = float(np.sum(times <= ta - gap))                # one window = one second of footage
    return exposure, sum(r <= ta - gap for r in runs), any(r < ta for r in runs)


def poisson_ci(k, conf=0.95):
    from scipy.stats import chi2
    a = 1 - conf
    lo = 0.0 if k == 0 else chi2.ppf(a / 2, 2 * k) / 2
    return lo, chi2.ppf(1 - a / 2, 2 * k + 2) / 2


def false_alarm_summary(per_clip):
    """per_clip: list of (exposure_s, runs, any_early). Runs per camera-hour with an exact Poisson CI."""
    exp_h = sum(e for e, _, _ in per_clip) / 3600.0; runs = sum(r for _, r, _ in per_clip)
    lo, hi = poisson_ci(runs)
    exps = sorted(e for e, _, _ in per_clip)
    return dict(clips=len(per_clip), exposure_hours=exp_h, runs=runs,
                runs_per_hour=runs / exp_h if exp_h else float("nan"),
                runs_per_hour_ci95=(lo / exp_h if exp_h else float("nan"), hi / exp_h if exp_h else float("nan")),
                clips_with_early_alarm=float(np.mean([a for _, _, a in per_clip])),
                clips_with_false_run=float(np.mean([r > 0 for _, r, _ in per_clip])),
                median_exposure_s=float(exps[len(exps) // 2]) if exps else 0.0)


def parity(a, b, rule, tol=0.05):
    """a, b: arrays of window scores for one clip from two paths. -> (share within tol, alarm decisions equal)."""
    a = np.asarray(a, float); b = np.asarray(b, float); m = min(len(a), len(b))
    same_len = len(a) == len(b)
    share = float(np.mean(np.abs(a[:m] - b[:m]) <= tol)) if m else 0.0
    t = np.arange(m)
    da = bool(alarm_runs(t, a[:m], rule["thr"], rule["k"], rule["n"])); db = bool(alarm_runs(t, b[:m], rule["thr"], rule["k"], rule["n"]))
    return share, da == db, same_len


In [ ]:
%%writefile m8_accident_video5.py
"""V5: V4 + hard-negative mining stage + K local crops + rise-over-baseline alarm rule (see TARGETS_VIDEO_V5_LOCKED.md).
Reuses the V3/V4 training and cache code; nothing here reads a test split before the single scoring pass."""
from __future__ import annotations
import json, pathlib, random
import numpy as np, pandas as pd, torch
import m8_accident_video as V
import m8_accident_video4 as V4
import m8_accident_train_v2 as A
import m8_accident_false_alarm as F

NF, SIZE = V.NF, V.SIZE
DELTA_GRID = [0.0, 0.15, 0.30]
MINE_GAP_S, MINE_MIN_P, MINE_MAX_PER_CLIP, HARD_PER_CLIP = 2.0, 0.4, 4, 2
STAGE2_EPOCHS, STAGE2_LR = 2, 2e-5
_ORIG_PROPOSALS = V4.proposals


# ------------------------------------------------------------ K local crops
def set_k(k):
    """Make every V4 function (training views, inference views, coverage) use `k` local crops."""
    V4.K_LOCAL = k
    V4.proposals = lambda heat, w, h, kk=None: _ORIG_PROPOSALS(heat, w, h, k if kk is None else kk)


def choose_k(train_df, cache, n=300, seed=0, min_gain=0.04):
    """K = 3 only if it lifts clip coverage by >= min_gain over K = 2 (measured on TRAIN clips)."""
    cov = {}
    for k in (2, 3):
        set_k(k); cov[k] = V4.proposal_coverage(train_df, cache, n=n, seed=seed)
    k = 3 if cov[3]["clip_has_covering_crop"] - cov[2]["clip_has_covering_crop"] >= min_gain else 2
    set_k(k); return k, cov


# ------------------------------------------------------------ alarm rule with a rise condition
def alarm_times(t, p, thr, k, n, delta=0.0):
    return [t[i] for i in range(len(p)) if F.hot_now(p, i, thr, k, n, delta)]


def evaluate5(probs, df, thr, k, n, delta):
    old = A.first_alarm_times
    A.first_alarm_times = lambda t, p, thr_, k_, n_: alarm_times(t, p, thr_, k_, n_, delta)
    try: return A.evaluate(probs, df, "path", thr, k, n)
    finally: A.first_alarm_times = old


def choose5(probs, df):
    best = None
    for k, n in A.CFG_GRID:
        for delta in DELTA_GRID:
            for thr in A.THR_GRID:
                m, _ = evaluate5(probs, df, thr, k, n, delta); feas = m["B2_early_alarm"] <= A.VAL_EARLY_CAP
                key = (feas, m["B1_detect_within_10s"] if feas else -m["B2_early_alarm"], -(m["B3_median_delay_s"] or 99), -delta)
                if best is None or key > best[0]: best = (key, {"thr": thr, "k": k, "n": n, "delta": delta, "val": m, "feasible": feas})
    return best[1]


# ------------------------------------------------------------ hard-negative mining + stage-2 data
@torch.no_grad()
def mine(model, df, cache, device=None, bs=12, log=print):
    """{path: [(window end index, view)]}: the pre-impact windows of TRAIN clips that the model scores highest, with their top view."""
    device = device or next(model.parameters()).device; model.eval(); hard = {}
    for _, r in df.iterrows():
        c = V4.Clip(cache[A.clip_key(r["path"])]); ta = float(r["accident_time"])
        ends = [e for e in V.window_ends(c.n()) if c.t[e] <= ta - MINE_GAP_S]
        items, meta = [], []
        for e in ends:
            for v in [None] + V4.proposals(c.heat(e), c.w, c.h): items.append(V4.make_view(c, e, v)); meta.append((e, v))
        ps = []
        for i in range(0, len(items), bs):
            with torch.autocast(device_type="cuda", enabled=str(device).startswith("cuda")):
                ps.extend(torch.sigmoid(V.logits(model, V.to_input(torch.from_numpy(np.stack(items[i:i + bs])), device)).float()).cpu().numpy().tolist())
        best = {}
        for (e, v), q in zip(meta, ps):
            if e not in best or q > best[e][0]: best[e] = (q, v)
        cand = sorted(((q, e, v) for e, (q, v) in best.items() if q >= MINE_MIN_P), key=lambda x: -x[0])[:MINE_MAX_PER_CLIP]
        if cand: hard[r["path"]] = [(int(e), v) for _, e, v in cand]
        c._fc.clear()
    log(f"  mined {sum(len(v) for v in hard.values())} hard negatives from {len(hard)}/{len(df)} train clips")
    return hard


class ClipDS5(torch.utils.data.Dataset):
    """V4 windows, but up to HARD_PER_CLIP of each clip's normal windows are its mined hard negatives."""
    def __init__(self, df, cache, hard, seed=0): self.df, self.cache, self.hard, self.seed = df.reset_index(drop=True), cache, hard, seed
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        empty = (torch.zeros(0, NF, SIZE, SIZE, 3, dtype=torch.uint8), torch.zeros(0))
        r = self.df.iloc[i]; c = V4.Clip(self.cache[A.clip_key(r["path"])]); ta = float(r["accident_time"]); rng = random.Random(hash((self.seed, i, random.random())))
        if c.n() < NF: return empty
        crash = (float(r["center_x"]), float(r["center_y"])); ws = V.sample_windows(c.t, ta, rng)
        pos = [w for w in ws if w[1] == 1]; neg = [w for w in ws if w[1] == 0]
        hard = self.hard.get(r["path"], []); h = rng.sample(hard, min(HARD_PER_CLIP, len(hard))) if hard else []
        neg = neg[:max(0, len(neg) - len(h))]
        xs, ys = [], []
        for e, y in pos + neg:
            locs = V4.proposals(c.heat(e), c.w, c.h)
            if y == 1:
                cov = [v for v in locs if V4.covers(v, crash, c.w, c.h)]; v = rng.choice(cov + [None]) if cov else None
            else: v = rng.choice([None] + locs)
            xs.append(V4.make_view(c, e, v)); ys.append(y)
        for e, v in h: xs.append(V4.make_view(c, e, v)); ys.append(0)
        return (torch.from_numpy(np.stack(xs)), torch.tensor(ys, dtype=torch.float32)) if xs else empty


def stage2(model, tr, cache, hard, seed=0, log=print):
    V.ClipDS = lambda df, cache_, seed_=0: ClipDS5(df, cache_, hard, seed_)
    try: return V.train(model, tr, cache, epochs=STAGE2_EPOCHS, lr=STAGE2_LR, log=log, seed=seed + 1)
    finally: V4.install()


# ------------------------------------------------------------ threshold-free extras + false-alarm proxy
def window_metrics(probs, df):
    from sklearn.metrics import average_precision_score, roc_auc_score
    y, s = [], []
    for _, r in df.iterrows():
        t, p = probs[r["path"]]; ta = float(r["accident_time"])
        for tt, pp in zip(t, p):
            lab = V.window_label(tt, ta)
            if lab >= 0: y.append(lab); s.append(float(pp))
    return {"windows": len(y), "positives": int(sum(y)), "AUROC": float(roc_auc_score(y, s)), "AP": float(average_precision_score(y, s))}


def false_alarm(probs, df, rule):
    return F.false_alarm_summary([F.pre_impact(*probs[r["path"]], float(r["accident_time"]), rule) for _, r in df.iterrows() if len(probs[r["path"]][0])])


# ------------------------------------------------------------ protocol
def run_protocol5(real, cache, split_col, out_dir, stage1=None, epochs=4, seed=0, log=print, model_fn=V.build_model):
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True); V4.install()
    tr_all = real[real[split_col] == "train"]; te = real[real[split_col] == "test"]
    tr, va = A.split_train_val(tr_all, "path", seed=seed); log(f"[{split_col}] train {len(tr)} val {len(va)} test {len(te)} | K={V4.K_LOCAL}")
    torch.manual_seed(seed)
    model = stage1 if stage1 is not None else V.train(model_fn(), tr, cache, epochs=epochs, log=log, seed=seed)     # stage 1 (V4 recipe)
    hard = mine(model, tr, cache, log=log)
    model = stage2(model, tr, cache, hard, seed=seed, log=log)
    cfg = choose5(V4.clip_probs(model, va, cache), va)
    log(f" CHOSEN on validation: thr {cfg['thr']} k/n {cfg['k']}/{cfg['n']} delta {cfg['delta']} feasible {cfg['feasible']} val {cfg['val']}")
    (out / f"chosen_v5_{split_col}.json").write_text(json.dumps({**cfg, "K_LOCAL": V4.K_LOCAL, "hard_negatives": sum(len(v) for v in hard.values())}, indent=1, default=str))
    probs = V4.clip_probs(model, te, cache)                                              # test scored ONCE
    m, d = evaluate5(probs, te, cfg["thr"], cfg["k"], cfg["n"], cfg["delta"])
    res = {"split": split_col, "thr": cfg["thr"], "k": cfg["k"], "n": cfg["n"], "delta": cfg["delta"], "K_LOCAL": V4.K_LOCAL, "val": cfg["val"], "test": m,
           "window_metrics": window_metrics(probs, te), "false_alarm": false_alarm(probs, te, cfg), "breakdown": A.breakdown(d, te, "path"),
           "hard_negatives": sum(len(v) for v in hard.values())}
    (out / f"results_v5_{split_col}.json").write_text(json.dumps(res, indent=1, default=str)); return res, model, probs


def verdict5(res_in, res_geo, auroc_ref_geo):
    a, g = res_in["test"], res_geo["test"]; bd = res_geo["breakdown"]["scene_layout"].get("highway", {"detect": 0})["detect"]
    fa = res_geo["false_alarm"]["runs_per_hour"]; au = res_geo["window_metrics"]["AUROC"]
    rows = [("I1 in-dist detect >= 72%", a["B1_detect_within_10s"], a["B1_detect_within_10s"] >= .72),
            ("I2 in-dist early <= 5%", a["B2_early_alarm"], a["B2_early_alarm"] <= .05),
            ("I3 in-dist strict >= 66%", a["strict_detect_no_early"], a["strict_detect_no_early"] >= .66),
            ("I4 in-dist median delay <= 5 s", a["B3_median_delay_s"], a["B3_median_delay_s"] is not None and a["B3_median_delay_s"] <= 5),
            ("G1 geo detect >= 64%", g["B1_detect_within_10s"], g["B1_detect_within_10s"] >= .64),
            ("G2 geo early <= 5%", g["B2_early_alarm"], g["B2_early_alarm"] <= .05),
            ("G3 geo strict >= 60%", g["strict_detect_no_early"], g["strict_detect_no_early"] >= .60),
            ("G4 geo highway detect >= 52%", bd, bd >= .52),
            ("G5 geo false-alarm proxy < 48 runs/h", fa, fa < 48)]
    adopt = rows[4][2] and rows[5][2] and rows[6][2] and au >= auroc_ref_geo
    return rows, {"geo_AUROC_v5": au, "geo_AUROC_v4": auroc_ref_geo, "ADOPT_V5": bool(adopt)}


In [ ]:
%%writefile m8_accident_video6.py
"""V6: ensemble of V5-recipe members (see TARGETS_VIDEO_V6_LOCKED.md).
Members record per-view window scores once (`dump`); the ensemble analysis (`analyse`) runs on CPU from those dumps and
touches the test split only for the one configuration chosen on validation."""
from __future__ import annotations
import json, pathlib, pickle, random
import numpy as np, pandas as pd, torch
import m8_accident_video as V
import m8_accident_video4 as V4
import m8_accident_train_v2 as A
import m8_accident_false_alarm as F
import m8_accident_video5 as V5

K = 3                                    # crops per window (V5's choice)
VAL_EARLY_CAP = 0.03                     # locked in TARGETS_VIDEO_V6_LOCKED.md
V5_PUBLISHED = {"thr": 0.85, "k": 3, "n": 5, "detect": 0.6764, "tol": 0.005}
REF = {"V4_proxy": 64.7, "V5_auroc": 0.815}


# ------------------------------------------------------------ per-view scores of one model
@torch.no_grad()
def dump(model, df, cache, device=None, bs=12, log=print):
    """{path: (window times, S)} with S (n_windows, 1 + K): global view first, then the motion-peak crops (NaN where a window has fewer)."""
    V4.install(); V5.set_k(K)
    device = device or next(model.parameters()).device; model.eval(); out = {}
    for j, (_, r) in enumerate(df.iterrows()):
        c = V4.Clip(cache[A.clip_key(r["path"])]); ends = V.window_ends(c.n()); items, owner, slot = [], [], []
        for w, e in enumerate(ends):
            for s, v in enumerate([None] + V4.proposals(c.heat(e), c.w, c.h)):
                items.append(V4.make_view(c, e, v)); owner.append(w); slot.append(s)
        S = np.full((len(ends), 1 + K), np.nan, np.float32)
        for i in range(0, len(items), bs):
            with torch.autocast(device_type="cuda", enabled=str(device).startswith("cuda")):
                q = torch.sigmoid(V.logits(model, V.to_input(torch.from_numpy(np.stack(items[i:i + bs])), device)).float()).cpu().numpy()
            for qq, w, s in zip(q, owner[i:i + bs], slot[i:i + bs]): S[w, s] = qq
        out[r["path"]] = (np.array([c.t[e] for e in ends]), S); c._fc.clear()
        if j % 200 == 0: log(f"  dumped {j}/{len(df)}")
    return out


def save_dump(d, path):
    with open(path, "wb") as f: pickle.dump(d, f, protocol=4)


def load_dump(path):
    with open(path, "rb") as f: return pickle.load(f)


# ------------------------------------------------------------ training one member (V5 recipe, different training seed)
def train_member(real, cache, train_seed, epochs=4, log=print, model_fn=V.build_model):
    V4.install(); V5.set_k(K)
    tr_all = real[real["split_geo_aware"] == "train"]
    tr, va = A.split_train_val(tr_all, "path", seed=0)           # SAME partition as V4 / V5 whatever the training seed
    torch.manual_seed(train_seed); random.seed(train_seed); np.random.seed(train_seed)
    model = V.train(model_fn(), tr, cache, epochs=epochs, log=log, seed=train_seed)
    hard = V5.mine(model, tr, cache, log=log)
    model = V5.stage2(model, tr, cache, hard, seed=train_seed, log=log)
    return model, tr, va


def member_dumps(model, real, cache, log=print):
    """Validation and test per-view scores of one member (nothing is thresholded here)."""
    tr_all = real[real["split_geo_aware"] == "train"]; te = real[real["split_geo_aware"] == "test"]
    _, va = A.split_train_val(tr_all, "path", seed=0)
    return {"val": dump(model, va, cache, log=log), "test": dump(model, te, cache, log=log)}


# ------------------------------------------------------------ analysis (CPU, from dumps)
def member_probs(d, df):
    return {p: (d[p][0], np.nanmax(d[p][1], axis=1).astype(np.float32) if len(d[p][0]) else np.zeros(0, np.float32)) for p in df["path"]}


def ensemble_probs(ds, df):
    mp = [member_probs(d, df) for d in ds]
    return {p: (mp[0][p][0], np.mean([m[p][1] for m in mp], axis=0).astype(np.float32)) for p in df["path"]}


def _score(probs, df, cfg):
    m, d = A.evaluate(probs, df, "path", cfg["thr"], cfg["k"], cfg["n"])
    return m, d


def choose(probs_val, va):
    old = A.VAL_EARLY_CAP; A.VAL_EARLY_CAP = VAL_EARLY_CAP
    try: return A.choose_config(probs_val, va, "path")
    finally: A.VAL_EARLY_CAP = old


def sanity_v5(dump_v5_test, te):
    """M0 with V5's published configuration must reproduce V5's published test detect rate."""
    m, _ = A.evaluate(member_probs(dump_v5_test, te), te, "path", V5_PUBLISHED["thr"], V5_PUBLISHED["k"], V5_PUBLISHED["n"])
    ok = abs(m["B1_detect_within_10s"] - V5_PUBLISHED["detect"]) <= V5_PUBLISHED["tol"]
    return ok, m["B1_detect_within_10s"]


def analyse(real, dumps, log=print):
    """dumps: {member name: {"val": dump, "test": dump}}. The test split is scored for the ONE chosen configuration (plus
    the per-member context rows, each with its own validation-chosen configuration)."""
    tr_all = real[real["split_geo_aware"] == "train"]; te = real[real["split_geo_aware"] == "test"]
    _, va = A.split_train_val(tr_all, "path", seed=0)
    names = sorted(dumps)
    if "M0" in dumps:
        ok, got = sanity_v5(dumps["M0"]["test"], te); log(f"sanity: M0 + published V5 config -> test detect {got:.4f} ({'OK' if ok else 'MISMATCH - result void'})")
        if not ok: raise RuntimeError("M0 dump does not reproduce V5's published test result; the dump is wrong")
    cfg = choose(ensemble_probs([dumps[n]["val"] for n in names], va), va)
    log(f"ensemble {names}: CHOSEN on validation thr {cfg['thr']} k/n {cfg['k']}/{cfg['n']} feasible {cfg['feasible']} val {cfg['val']}")
    probs = ensemble_probs([dumps[n]["test"] for n in names], te)               # test scored ONCE for the chosen config
    m, d = _score(probs, te, cfg)
    res = {"members": names, "thr": cfg["thr"], "k": cfg["k"], "n": cfg["n"], "val": cfg["val"], "feasible": cfg["feasible"], "test": m,
           "window_metrics": V5.window_metrics(probs, te), "false_alarm": V5.false_alarm(probs, te, cfg), "breakdown": A.breakdown(d, te, "path")}
    ctx = {}
    for n in names:                                                              # context only
        c = choose(member_probs(dumps[n]["val"], va), va); mm, _ = _score(member_probs(dumps[n]["test"], te), te, c)
        ctx[n] = {"thr": c["thr"], "k": c["k"], "n": c["n"], "test": mm}
    res["members_alone"] = ctx
    return res


def verdict6(res):
    t = res["test"]; fa = res["false_alarm"]["runs_per_hour"]; au = res["window_metrics"]["AUROC"]
    hw = res["breakdown"]["scene_layout"].get("highway", {"detect": float("nan")})["detect"]
    rows = [("E1 geo detect >= 66%", t["B1_detect_within_10s"], t["B1_detect_within_10s"] >= .66),
            ("E2 geo early <= 5%", t["B2_early_alarm"], t["B2_early_alarm"] <= .05),
            ("E3 geo strict >= 61%", t["strict_detect_no_early"], t["strict_detect_no_early"] >= .61),
            ("E4 geo false-alarm proxy <= 64.7 runs/h", fa, fa <= REF["V4_proxy"]),
            ("E5 geo window AUROC >= 0.815", au, au >= REF["V5_auroc"])]
    return rows, {"adopt_ensemble": all(r[2] for r in rows), "highway_detect_reported": hw}


In [ ]:
%%writefile m8_accident_video7.py
"""V7: V5 recipe started from the self-supervised VideoMAE-base (see TARGETS_VIDEO_V7_LOCKED.md).
Nothing new is modelled here: the backbone comes from the ACC_VID environment variable (set BEFORE importing m8_accident_video),
stage 1 is 6 epochs, everything else is run_protocol5 (hard-negative stage, K=3 crops, delta grid, validation-chosen alarm, test scored once)."""
from __future__ import annotations
import os
import m8_accident_video as V
import m8_accident_video4 as V4
import m8_accident_video5 as V5
import m8_accident_video6 as V6

BACKBONE = "MCG-NJU/videomae-base"
K = 3                      # V5's measured choice, not re-chosen
STAGE1_EPOCHS = 6          # the one locked deviation from V5 (4)


def check_backbone(model_id=None):
    """Void-the-run guard: the encoder and embedding weights must load from the checkpoint (only the new head / fc_norm may be missing)."""
    from transformers import VideoMAEForVideoClassification
    mid = model_id or V.MODEL_ID
    _, info = VideoMAEForVideoClassification.from_pretrained(mid, num_labels=1, ignore_mismatched_sizes=True, output_loading_info=True)
    missing = sorted(info.get("missing_keys", []))
    bad = [k for k in missing if "encoder" in k or "embeddings" in k]
    if bad: raise RuntimeError(f"{mid}: encoder / embedding weights NOT loaded: {bad[:5]} ... ({len(bad)} keys); result would be void")
    return {"model_id": mid, "missing_keys": missing, "unexpected": len(info.get("unexpected_keys", [])), "mismatched": len(info.get("mismatched_keys", []))}


def run7(real, cache, out_dir, epochs=STAGE1_EPOCHS, log=print, model_fn=None):
    V4.install(); V5.set_k(K)
    return V5.run_protocol5(real, cache, "split_geo_aware", out_dir, stage1=None, epochs=epochs, seed=0, log=log, model_fn=model_fn or V.build_model)


def verdict7(res):
    rows, extra = V6.verdict6(res)
    rows = [(r[0].replace("E", "H", 1), r[1], r[2]) for r in rows]
    return rows, {"adopt_v7": bool(extra["adopt_ensemble"]), "highway_detect_reported": extra["highway_detect_reported"]}


In [ ]:
import sys, pathlib, json, os, pandas as pd, numpy as np, torch
assert os.environ.get('ACC_VID') == 'MCG-NJU/videomae-base'
sys.path.insert(0, '.'); import m8_accident_video7 as V7, m8_accident_video5 as V5, m8_accident_video4 as V4, m8_accident_video as V, m8_accident_train_v2 as A
assert V.MODEL_ID == 'MCG-NJU/videomae-base', V.MODEL_ID
ROOT = pathlib.Path(ROOT_DIR); real = pd.read_csv(ROOT/'metadata-real.csv')
OUT = '/kaggle/working/out_v7'; os.makedirs(OUT, exist_ok=True)
info = V7.check_backbone(); print(info); json.dump(info, open(f'{OUT}/backbone_check.json', 'w'), indent=1)   # raises if encoder weights did not load
cache = V4.extract_all([(ROOT/p, A.clip_key(p)) for p in real['path']], '/tmp/cache_j960')
print(len(cache), 'clips cached')

In [ ]:
# Stage 1 (6 epochs) -> hard-negative mining -> stage 2 -> alarm rule chosen on validation -> geo test scored ONCE
res, model, probs = V7.run7(real, cache, OUT)
print(res['test'], res['window_metrics'], res['false_alarm'])
import transformers
sd = {k: (v.half() if v.is_floating_point() else v).cpu() for k, v in model.state_dict().items()}
torch.save(sd, f'{OUT}/itms_accident_v7.pt')
cfg = json.load(open(f'{OUT}/chosen_v5_split_geo_aware.json'))
json.dump({'thr': cfg['thr'], 'k': cfg['k'], 'n': cfg['n'], 'delta': cfg['delta'], 'K_LOCAL': cfg['K_LOCAL'], 'split': 'geo_aware', 'val': cfg.get('val'), 'backbone': V.MODEL_ID,
           'transformers': transformers.__version__, 'torch': torch.__version__}, open(f'{OUT}/itms_accident_v7.json', 'w'), indent=1)

In [ ]:
rows, summ = V7.verdict7(res)
print('| target | result | pass |'); print('|---|---|---|')
for name, val, ok in rows: print(f'| {name} | {val if val is None else round(val, 3)} | {"PASS" if ok else "FAIL"} |')
print(summ)
json.dump({'rows': rows, **summ}, open(f'{OUT}/verdict_v7.json', 'w'), indent=1, default=str)
print(json.dumps(res['breakdown'], indent=1))

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/accident_v7_results', 'zip', OUT); print('saved /kaggle/working/accident_v7_results.zip')